# 02 — Extract Item 9A

Read filing HTML from `raw/`, extract the Item 9A span, write records to `processed/`.

**Rejects rather than guesses.** A filing whose Item 9A cannot be located goes to the extraction-failure list with a reason. A truncated or whole-document fallback never enters `processed/`.

Read the `edgar-harvesting` skill before changing anything here.

In [ ]:
# Colab bootstrap. Run once per runtime.
!pip install -q google-cloud-storage beautifulsoup4 jsonschema

from google.colab import auth
auth.authenticate_user()

import sys, pathlib
GITHUB_USER = ''   # TODO
REPO = pathlib.Path('/content/auditagent-bench')
if not REPO.exists():
    !git clone -q https://github.com/{GITHUB_USER}/auditagent-bench.git {REPO}
sys.path.insert(0, str(REPO))

In [ ]:
# --- Config. Every tunable value in this notebook lives in this cell. ---
BUCKET = ''            # TODO: GCS bucket name
RAW_PREFIX = 'raw'
PROCESSED_PREFIX = 'processed'

In [ ]:
from src import gcs, parsing, schema
from src.parsing import ExtractionFailure
from datetime import datetime, timezone

raw_blobs = [b for b in gcs.list_prefix(BUCKET, RAW_PREFIX) if b.endswith('.html')]
print(f'{len(raw_blobs)} filings in {RAW_PREFIX}/')

In [ ]:
records, failures = [], []

for blob in raw_blobs:
    accession = blob.split('/')[-1].removesuffix('.html')
    meta = gcs.read_json(BUCKET, f'{RAW_PREFIX}/meta/{accession}.json')
    try:
        text = parsing.extract_item_9a(gcs.read_text(BUCKET, blob))
    except ExtractionFailure as exc:
        failures.append({'accession_number': accession, 'reason': str(exc)})
        continue

    records.append({
        **meta,
        'fiscal_year': int(meta['filing_date'][:4]),
        'item_9a_text': text,
        'extraction_method': 'heading_to_next_item',
        'fetched_at': datetime.now(timezone.utc).isoformat(),
    })

print(f'extracted={len(records)} failed={len(failures)}')
if raw_blobs:
    print(f'failure rate {len(failures) / len(raw_blobs):.1%} — investigate above ~10%')

In [ ]:
# Validate against the schema before writing. A silently degraded corpus is the
# main quality risk in this stage.
errors = schema.validate_records(records, 'filing_record')

if errors:
    print(f'{len(errors)} schema errors — nothing written')
    for e in errors[:10]:
        print(' ', e)
else:
    n = gcs.write_jsonl(BUCKET, f'{PROCESSED_PREFIX}/item9a.jsonl', records)
    gcs.write_json(BUCKET, f'{PROCESSED_PREFIX}/_extraction_failures.json', failures)
    print(f'wrote {n} records')